In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1))

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [6]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('bird',
   -0.08544646425023128,
   tensor(10.4561, grad_fn=<SubBackward0>),
   tensor(8.0728, grad_fn=<SubBackward0>),
   tensor(-31.0837, grad_fn=<AddBackward0>),
   tensor(17.8235, grad_fn=<AddBackward0>)),
  ('bird',
   -0.10037175803220588,
   tensor(28.8584, grad_fn=<SubBackward0>),
   tensor(-29.2646, grad_fn=<SubBackward0>),
   tensor(-52.0216, grad_fn=<AddBackward0>),
   tensor(20.6438, grad_fn=<AddBackward0>)),
  ('dog',
   -0.07786106302489948,
   tensor(18.8739, grad_fn=<SubBackward0>),
   tensor(-7.6950, grad_fn=<SubBackward0>),
   tensor(50.0311, grad_fn=<AddBackward0>),
   tensor(-5.0987, grad_fn=<AddBackward0>)),
  ('dog',
   0.10254990739435321,
   tensor(60.0346, grad_fn=<SubBackward0>),
   tensor(18.6005, grad_fn=<SubBackward0>),
   tensor(32.2354, grad_fn=<AddBackward0>),
   tensor(-13.6073, grad_fn=<AddBackward0>)),
  ('sheep',
   -0.008839716851268609,
   tensor(74.1016, grad_fn=<SubBackward0>),
   tensor(-37.1776, grad_fn=<SubBackward0>),
   tensor(47.4273, gr

In [7]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{'dog': [('dog',
    0.48632621735943005,
    tensor(-5.9985, grad_fn=<SubBackward0>),
    tensor(168.9849, grad_fn=<SubBackward0>),
    tensor(-32.2500, grad_fn=<AddBackward0>),
    tensor(150.9992, grad_fn=<AddBackward0>))]},
 {},
 {},
 {},
 {},
 {'bird': [('bird',
    0.4837727426861136,
    tensor(62.8674, grad_fn=<SubBackward0>),
    tensor(71.1385, grad_fn=<SubBackward0>),
    tensor(131.3517, grad_fn=<AddBackward0>),
    tensor(-75.6001, grad_fn=<AddBackward0>)),
   ('bird',
    0.4928229227326142,
    tensor(215.9692, grad_fn=<SubBackward0>),
    tensor(-31.7910, grad_fn=<SubBackward0>),
    tensor(167.8921, grad_fn=<AddBackward0>),
    tensor(175.4803, grad_fn=<AddBackward0>)),
   ('bird',
    0.6508499837931083,
    tensor(8.4981, grad_fn=<SubBackward0>),
    tensor(140.6253, grad_fn=<SubBackward0>),
    tensor(106.6098, grad_fn=<AddBackward0>),
    tensor(108.9426, grad_fn=<AddBackward0>))],
  'bottle': [('bottle',
    0.6311955132945002,
    tensor(133.1648, grad_fn=<SubBa

In [8]:
import torch 
nums = torch.arange(10)
nums = nums.sort()[0]

print(nums)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [9]:
nums = [num for num in nums if num == 2 or num == 4 or num == 6]
nums

[tensor(2), tensor(4), tensor(6)]

In [10]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

    final_preds.append(final_img_preds)

In [11]:
final_preds, len(final_preds)

([{'dog': [('dog',
     0.48632621735943005,
     tensor(-5.9985, grad_fn=<SubBackward0>),
     tensor(168.9849, grad_fn=<SubBackward0>),
     tensor(-32.2500, grad_fn=<AddBackward0>),
     tensor(150.9992, grad_fn=<AddBackward0>))]},
  {},
  {},
  {},
  {},
  {'bird': [('bird',
     0.4837727426861136,
     tensor(62.8674, grad_fn=<SubBackward0>),
     tensor(71.1385, grad_fn=<SubBackward0>),
     tensor(131.3517, grad_fn=<AddBackward0>),
     tensor(-75.6001, grad_fn=<AddBackward0>)),
    ('bird',
     0.4928229227326142,
     tensor(215.9692, grad_fn=<SubBackward0>),
     tensor(-31.7910, grad_fn=<SubBackward0>),
     tensor(167.8921, grad_fn=<AddBackward0>),
     tensor(175.4803, grad_fn=<AddBackward0>)),
    ('bird',
     0.6508499837931083,
     tensor(8.4981, grad_fn=<SubBackward0>),
     tensor(140.6253, grad_fn=<SubBackward0>),
     tensor(106.6098, grad_fn=<AddBackward0>),
     tensor(108.9426, grad_fn=<AddBackward0>))],
   'bottle': [('bottle',
     0.6311955132945002,
     

In [12]:
y_batch.shape, y_batch, y_batch[0].shape

(torch.Size([32, 7, 7, 30]),
 tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.00

In [13]:
(y_batch[0].flatten(0, 1))[29]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.])

In [15]:
from src.configs import IDX_TO_CLASS

def find_objects(y_batch, device):
    y_batch = y_batch.flatten(1, 2)
    truth_objects = []

    for target in y_batch:
        class_objects = {}
        
        for cell in target:
            class_name = IDX_TO_CLASS[int(torch.argmax(cell[:C]))]

            if cell[C+4] == 1:
                # the additional 0 is to classify that specific object as 
                # unmatched with a prediction. 1 is for matched. 
                bboxes = torch.cat((cell[C:C+4], torch.tensor([0]).to(device)))
                if class_name in class_objects:
                    class_objects[class_name].append(bboxes)
                else:
                    class_objects[class_name] = [bboxes]

        truth_objects.append(class_objects)
        
    return truth_objects

truth_objects = find_objects(y_batch, "cpu")

In [16]:
len(truth_objects)

32

In [17]:
final_preds[0]

{'dog': [('dog',
   0.48632621735943005,
   tensor(-5.9985, grad_fn=<SubBackward0>),
   tensor(168.9849, grad_fn=<SubBackward0>),
   tensor(-32.2500, grad_fn=<AddBackward0>),
   tensor(150.9992, grad_fn=<AddBackward0>))]}

In [19]:
TP_IOU_THRESHOLD = 0.5

def find_tp_fp(final_preds, truth_objects, all_tp_fp_by_class):
    # 1. iterate through each image prediction/label in the batch
    for b in range(len(final_preds)):
        img_preds = final_preds[b]
        objects = truth_objects[b]

        # 2. iterate through each class
        for class_name, preds in img_preds.items():            
            # 3. if any of the ground truth objects belong to the class
            if class_name in objects:
                # iterate through each prediction, find max IoU truth object, and
                # if the max IoU surpasses the threshold, it is a TP, else FP
                for pred in preds:
                    IoUs = [IoU(object_[0:4], pred[2:6]) for object_ in objects[class_name]]
                    max_idx = IoUs.index(max(IoUs))
                    
                    if IoUs[max_idx] < TP_IOU_THRESHOLD:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
                        
                    elif objects[class_name][max_idx][-1] == 0:
                        all_tp_fp_by_class[class_name].append((pred[1], True))
                        objects[class_name][max_idx][-1] = 1
                        
                    else:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
            else:
                # all predictions belonging to class are FP since there are
                # no ground truth objects belonging to that class
                for pred in preds:
                    all_tp_fp_by_class[class_name].append((pred[1], False))    

In [20]:
all_tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow":[],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

In [24]:
find_tp_fp(final_preds, truth_objects, all_tp_fp_by_class)

all_tp_fp_by_class["bird"].sort(key=itemgetter(0), reverse=True)
all_tp_fp_by_class

{'aeroplane': [(0.7013003062053826, False),
  (0.5259327844376358, False),
  (0.7013003062053826, False),
  (0.5259327844376358, False),
  (0.7013003062053826, False),
  (0.5259327844376358, False),
  (0.7013003062053826, False),
  (0.5259327844376358, False)],
 'bicycle': [(0.5045754901505752, False),
  (0.48202983623834683, False),
  (0.5045754901505752, False),
  (0.48202983623834683, False),
  (0.5045754901505752, False),
  (0.48202983623834683, False),
  (0.5045754901505752, False),
  (0.48202983623834683, False)],
 'bird': [(0.6508499837931083, False),
  (0.6508499837931083, False),
  (0.6508499837931083, False),
  (0.6508499837931083, False),
  (0.5468570412325775, False),
  (0.5468570412325775, False),
  (0.5468570412325775, False),
  (0.5468570412325775, False),
  (0.4928229227326142, False),
  (0.4928229227326142, False),
  (0.4928229227326142, False),
  (0.4928229227326142, False),
  (0.4837727426861136, False),
  (0.4837727426861136, False),
  (0.4837727426861136, False),
 